# Telcovantage — GPU TrOCR Server (Cloudflare Tunnel)

Runs TrOCR models on Colab GPU and exposes them via Cloudflare Tunnel.

**Workflow:**
1. Run all cells below
2. Copy the `REMOTE_TROCR_URL` printed at the end
3. On your laptop: `$env:REMOTE_TROCR_URL = "<url>"` then `py server.py`

---
## Cell 1: Install dependencies

In [ ]:
!pip install -q fastapi uvicorn

---
## Cell 2: Install Cloudflare Tunnel

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!cloudflared --version

---
## Cell 3: Clone your repo

In [ ]:
import os
REPO_URL = "https://github.com/jonrenzo/Telcovantage-Site-Map-Reader"
REPO_DIR = "/content/Telcovantage-Site-Map-Reader"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull

---
## Cell 4: Install project Python dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt
print("Dependencies installed.")

---
## Cell 5: Start TrOCR server (non-blocking)

In [ ]:
import threading, time, sys, requests
sys.path.insert(0, REPO_DIR)

import uvicorn
from colab.server import app

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

t = threading.Thread(target=run_server, daemon=True)
t.start()

# Wait for server to start
for i in range(30):
    time.sleep(1)
    try:
        r = requests.get("http://localhost:8000/health", timeout=3)
        if r.status_code == 200:
            print("Server started on port 8000.")
            break
    except Exception:
        pass

# Pre-warm the model
print("Loading TrOCR model on GPU (first run may take ~15-30s)...")
resp = requests.post("http://localhost:8000/ocr/pole",
    json={"segments": [{"x1":0,"y1":0,"x2":10,"y2":0},
                      {"x1":5,"y1":0,"x2":5,"y2":20}],
          "bbox": [0,0,10,20], "auto_rotate": False}, timeout=180)
print("Model loaded:", resp.status_code)

---
## Cell 6: Start Cloudflare Tunnel (gives public URL)

In [ ]:
# This will print a URL like https://xxxx.trycloudflare.com
# Keep this cell running to keep the tunnel alive.
!cloudflared tunnel --url http://localhost:8000

---
## Cell 7: Keep-alive (non-blocking)

In [ ]:
from IPython.display import display, Javascript
display(Javascript("""
if (!window._colabKeepalive) {
  window._colabKeepalive = setInterval(function(){}, 60000);
  console.log('Keepalive started.');
}
"""))
print("Keepalive active (browser JS, does not block kernel).")